# All mispredictions for one gloss

| EXP NO    | Model    | Split            |
|-----------|----------|------------------|
| exp000    | MViTv2_S | asl2000_cutoff_9 |

Follow-up to [eda_performers.ipynb](./eda_performers.ipynb), which only shows each
underachiever's *most common* misprediction (e.g. "before" as "former", 1/4 wrong
instances). Here one gloss is picked and every one of its instances is shown with the
model's top-k confidences, then the frames of whatever it was predicted as can be
stepped through for comparison.

Dataset fields, split/set naming and gotchas: [WLASL_info.md](../../info/WLASL_info.md).

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import HTML, display
from pydantic import TypeAdapter

from src.configs import get_class_list
from src.preprocess import Instance
from src.results import (
    format_exp_label,
    get_asset_path,
    get_out_stub,
    get_stash_path,
    load_json,
)
from src.run_types import AVAIL_SETS, CUTOFF_SPLITS, InstanceTopK
from src.stats import get_all_sets
from src.visualise2 import (
    BarPosition,
    FrameFetcher,
    animate_frames_topk,
    plot_frame_grid,
    plot_frame_grid_topk,
    save_fig,
    set_thesis_style,
)

## Setup

In [ ]:
set_thesis_style()

split_name: CUTOFF_SPLITS = 'asl2000_cutoff_9'
set_name: AVAIL_SETS = 'test' # the set the predictions were stashed for
model_name = 'MViTv2_S'
exp_no = '000'
checkpoint_num = None # override if multiple runs of the same experiment
project_name = 'satnac_2026'

gloss = 'before' # change to inspect another gloss
pred_set_name: AVAIL_SETS = 'test' # set to browse the predicted gloss's instances from
k = 5 # bars per top-k chart, <= max_k below
bar_position: BarPosition = 'right'
shared_axis = True # same 0-1 confidence axis on every chart, so instances compare
target_length = 16
grid_cell_size = (2.0, 2.0)
save_assets = False # save each viewed figure to results/outputs

stub = get_out_stub(split_name, model_name, format_exp_label(exp_no), checkpoint_num)
classes = get_class_list()
cls_idx = classes.index(gloss)
all_sets = get_all_sets(split_name)

## Per-instance top-k predictions

The stash from `correlation_f1.ipynb` only has each instance's top-1 prediction, so the
confidences are inferred separately. This needs the trained checkpoint, so run it once on
the training server, then leave the call commented out and load the stash below.

In [ ]:
topk_stash_p = get_stash_path(stub, base_name='instance_topk')
max_k = 20 # how many classes are stored per instance -- caps `k`
instance_topk_adapter = TypeAdapter(list[InstanceTopK])


def run_topk_inference() -> None:
    from src.results import stash_json
    from src.run_types import MinInfo
    from src.testing import get_model_checkpoint_dir, get_model_exp_dir
    from src.visualise2 import infer_instance_topk

    output = get_model_exp_dir(split=split_name, model=model_name, exp_no=int(exp_no))
    save_path = get_model_checkpoint_dir(output, checkpoint_num)
    admin = MinInfo(model=model_name, split=split_name, save_path=str(save_path))
    results = infer_instance_topk(admin=admin, set_name=set_name, split_name=split_name, max_k=max_k)
    stash_json(instance_topk_adapter.dump_python(results), topk_stash_p)
    print(f'Saved to: {topk_stash_p}')

# run_topk_inference()

In [ ]:
predictions = {
    p.video_id: p for p in instance_topk_adapter.validate_python(load_json(topk_stash_p))
}
print(f'Loaded top-{max_k} predictions for {len(predictions)} {set_name} instances')

## Every prediction for the gloss

`True rank` is the gloss's position in the model's ranking (`>max_k` if it isn't stored).

In [ ]:
def top1_gloss(pred: InstanceTopK) -> str:
    return classes[pred.topk_idxs[0]]


def true_rank(pred: InstanceTopK) -> str:
    return str(pred.topk_idxs.index(pred.target) + 1) if pred.target in pred.topk_idxs else f'>{max_k}'


gloss_instances = [Instance.model_validate(i) for i in all_sets[set_name][cls_idx]['instances']]
gloss_preds = [predictions[inst.video_id] for inst in gloss_instances]
summary_df = pd.DataFrame(
    {
        'Video ID': [inst.video_id for inst in gloss_instances],
        'Signer': [inst.signer_id for inst in gloss_instances],
        'Predicted': [top1_gloss(p) for p in gloss_preds],
        'Confidence': [p.topk_probs[0] for p in gloss_preds],
        'True rank': [true_rank(p) for p in gloss_preds],
    }
)
print(f'"{gloss}": {len(gloss_instances)} {set_name} instances')
display(Counter(summary_df['Predicted']).most_common())
summary_df

## Step through the gloss's instances

Run the first cell once, then re-run the second to show the next instance (wraps around
after the last). Each shows the clip with its top-k confidences, the true gloss's bar in
green when it's among them, and sets `predicted_gloss` for the section below.

In [ ]:
gloss_fetcher = FrameFetcher(
    cycle=True,
    cls_idx=cls_idx,
    all_sets=all_sets,
    set_name=set_name,
    split_name=split_name,
    target_length=target_length,
)
pred_fetcher: FrameFetcher | None = None # built by the predicted-gloss viewer below
pred_fetcher_gloss: str | None = None

In [ ]:
frames = gloss_fetcher()
inst = gloss_fetcher.current_instance
pred = predictions[inst.video_id]
predicted_gloss = top1_gloss(pred)

print(f'Instance {gloss_fetcher.cur_idx}/{gloss_fetcher.len}: video {inst.video_id}, signer {inst.signer_id}')
print(f'Predicted "{predicted_gloss}" ({pred.topk_probs[0]:.2f}); "{gloss}" ranked {true_rank(pred)}')
fig, _, _ = plot_frame_grid_topk(
    frames,
    labels=[classes[i] for i in pred.topk_idxs],
    scores=pred.topk_probs,
    k=k,
    true_label=gloss,
    bar_position=bar_position,
    shared_axis=shared_axis,
    num=target_length,
    size=grid_cell_size,
)
if save_assets:
    print(f'Saved: {save_fig(fig, get_asset_path(f"topk_{gloss}_{inst.video_id}", stub, project_name)).name}')

Optionally, the same instance as a video. These are the `target_length` frames the grid
was sampled from, not the full clip.

In [ ]:
anim_fig, anim, _ = animate_frames_topk(
    frames,
    labels=[classes[i] for i in pred.topk_idxs],
    scores=pred.topk_probs,
    k=k,
    true_label=gloss,
    bar_position=bar_position,
    shared_axis=shared_axis,
    interval=150,
)
plt.close(anim_fig) # otherwise the static first frame is shown as well
display(HTML(anim.to_jshtml()))

## Step through the predicted gloss's instances

Re-run to show the next instance of the current `predicted_gloss`. It restarts from that
gloss's first instance whenever the cell above moves to an instance predicted as
something else. Instances from `set_name` also get their own top-k chart.

In [ ]:
if pred_fetcher is None or pred_fetcher_gloss != predicted_gloss:
    pred_fetcher_gloss = predicted_gloss
    pred_fetcher = FrameFetcher(
        cycle=True,
        cls_idx=classes.index(predicted_gloss),
        all_sets=all_sets,
        set_name=pred_set_name,
        split_name=split_name,
        target_length=target_length,
    )

if pred_fetcher.len == 0:
    print(f'"{predicted_gloss}" has no {pred_set_name} instances')
else:
    pred_frames = pred_fetcher()
    pred_inst = pred_fetcher.current_instance
    print(f'"{predicted_gloss}" instance {pred_fetcher.cur_idx}/{pred_fetcher.len}: video {pred_inst.video_id}, signer {pred_inst.signer_id}')
    if pred_set_name == set_name:
        other_pred = predictions[pred_inst.video_id]
        fig, _, _ = plot_frame_grid_topk(
            pred_frames,
            labels=[classes[i] for i in other_pred.topk_idxs],
            scores=other_pred.topk_probs,
            k=k,
            true_label=predicted_gloss,
            bar_position=bar_position,
            shared_axis=shared_axis,
            num=target_length,
            size=grid_cell_size,
        )
    else:
        fig, _ = plot_frame_grid(pred_frames, num=target_length, size=grid_cell_size)